# Building a CIM Network Model by Hand

This tutorial builds a small power system network model **from scratch** using
CIMantic Graphs (`cimgraph`). By the end you will have assembled a complete,
exportable CIM model containing every major building block:

| Component | CIM class | Role |
|-----------|-----------|------|
| Grid connection | `EnergySource` | The "slack" / infinite source |
| Lines | `ACLineSegment` | Branches between buses |
| Load | `EnergyConsumer` | A customer load |
| Switch | `LoadBreakSwitch` | Sectionalizing / isolation device |
| Capacitor | `LinearShuntCompensator` | Shunt capacitor for voltage support |
| Battery inverter | `PowerElectronicsConnection` + `BatteryUnit` | A DER |
| Buses | `ConnectivityNode` | Electrical nodes |
| Connections | `Terminal` | How equipment attaches to buses |
| Containers | `Feeder`, `Substation` | Organize and name the model |

### The network we will build

```
[EnergySource]                                              [Switch]        [Battery inverter]
  "grid"                                                     "sw_4_5"         (PEC + BatteryUnit)
     |                                                          |                     |
   bus_1 --(line_1_2)-- bus_2 --(line_2_3)-- bus_3 --(line_3_4)-- bus_4 --/ --  bus_5
                          |                    |
                    [EnergyConsumer]      [Capacitor]
                        "load_a"            "cap_3"
```

A 5-bus radial feeder: the grid feeds `bus_1`; three line segments carry power
down to `bus_4`; a load taps off `bus_2`; a shunt capacitor supports voltage at
`bus_3`; and a `LoadBreakSwitch` connects `bus_4` to `bus_5`, where a
battery-backed inverter sits. The switch lets us isolate the DER from the rest of
the feeder.

## 1. Setup

Select the CIM profile **before** importing it. The profile defines which CIM
classes and attributes are available. We use `cimhub_2026`.

In [1]:
import os
os.environ['CIMG_CIM_PROFILE'] = 'cimhub_2026'
os.environ['CIMG_USE_UNITS'] = 'TRUE'   # enable pint-backed CIMUnit values

import cimgraph.data_profile.cimhub_2026 as cim
from cimgraph.models import FeederModel
from cimgraph.databases import XMLFile
from cimgraph import utils
from mermaid import Mermaid

## 2. Create the network graph

Everything we build must be registered in a **graph** so it can be queried and
exported. The graph is backed by a connection — here an `XMLFile` that we will
write to at the end.

> The two warnings below are expected: the file doesn't exist yet, so cimgraph
> starts from an empty graph.

In [2]:
file = XMLFile('hand_built_model.xml')
network = FeederModel(container=None, connection=file)

Each object we create gets added with `network.add_to_graph(obj)`. Forgetting
this is the most common mistake — an object that isn't in the graph won't be
exported.

### The class hierarchy we'll use

Before building anything, here is the **UML class diagram** for the equipment
types in this tutorial. Almost everything that carries current inherits from
`ConductingEquipment`. `cimgraph.utils.get_mermaid()` renders these straight from
the profile definitions.

In [3]:
diagram = utils.get_mermaid([
    cim.ConductingEquipment,   # anything that conducts electricity
    cim.ACLineSegment,         # lines and cables
    cim.EnergySource,          # the grid connection / slack source
    cim.EnergyConsumer,        # loads
    cim.LoadBreakSwitch,       # switches
    cim.LinearShuntCompensator,  # shunt capacitors
    cim.PowerElectronicsConnection,  # inverters
], show_attributes=False)
Mermaid(diagram)

## 3. Containers: Feeder and Substation

CIM organizes equipment into **containers**. Every piece of equipment points to
its `EquipmentContainer`, and every node points to its `ConnectivityNodeContainer`.
For a distribution model the top-level container is a `Feeder`, energized by a
`Substation`.

In [4]:
feeder = cim.Feeder(name='demo_feeder')
network.add_to_graph(feeder)

substation = cim.Substation(name='demo_sub')
network.add_to_graph(substation)

# A Feeder is normally energized by one Substation
feeder.NormalEnergizingSubstation = substation
substation.NormalEnergizedFeeder.append(feeder)  # optional reverse association

feeder.pprint()

{
    "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
    "@type": "Feeder",
    "name": "demo_feeder",
    "NormalEnergizingSubstation": {
        "@id": "6a7a1743-534a-48a0-9216-2b7f9139797d",
        "@type": "Substation"
    }
}


## 4. Base voltage

A `BaseVoltage` defines the nominal voltage of a section of the network. We
build a single 12.47 kV base and share it across all the equipment.

Note the use of `cim.Voltage(...)` — **never** multiply by 1000 by hand. The
`CIMUnit` constructor takes the value and its unit and stores it internally in
SI base units (volts).

In [5]:
base_12kv = cim.BaseVoltage(name='base_12.47kV')
base_12kv.nominalVoltage = cim.Voltage(12.47, 'kV')
network.add_to_graph(base_12kv)

print(base_12kv.nominalVoltage)            # stored in volts
print(base_12kv.nominalVoltage.to('kV'))   # convert back for display

12470.0 volt
12.47


## 5. Buses (ConnectivityNodes)

A `ConnectivityNode` is an electrical bus — the point where terminals of
equipment meet. We create four of them. Each is contained in the feeder.

In [6]:
bus_names = ['bus_1', 'bus_2', 'bus_3', 'bus_4', 'bus_5']
buses = {}

for name in bus_names:
    node = cim.ConnectivityNode(name=name, ConnectivityNodeContainer=feeder)
    network.add_to_graph(node)
    buses[name] = node

print(list(buses))

['bus_1', 'bus_2', 'bus_3', 'bus_4', 'bus_5']


## 6. A reusable connection helper

Every piece of `ConductingEquipment` connects to a bus through a `Terminal`.
The pattern is always the same:

1. Create a `Terminal` with a `sequenceNumber` (1 for the first/only end, 2 for
   the far end of a branch).
2. Point the terminal at its equipment (`ConductingEquipment`) and its bus
   (`ConnectivityNode`).
3. Append it to both reverse lists so the graph is navigable in both directions.

Rather than repeat this six times, we wrap it in a small helper.

The UML diagram below shows the relationship at the heart of CIM connectivity:
a `ConductingEquipment` has `Terminals`, and each `Terminal` references one
`ConnectivityNode`.

In [7]:
diagram = utils.get_mermaid([
    cim.ConductingEquipment,
    cim.Terminal,
    cim.ConnectivityNode,
], show_attributes=False)
Mermaid(diagram)

In [8]:
def connect(equipment, bus, sequence_number):
    """Attach `equipment` to `bus` via a new Terminal and register it."""
    terminal = cim.Terminal(
        name=f'{equipment.name}_t{sequence_number}',
        sequenceNumber=sequence_number,
    )
    terminal.ConductingEquipment = equipment
    terminal.ConnectivityNode = bus
    equipment.Terminals.append(terminal)
    bus.Terminals.append(terminal)
    network.add_to_graph(terminal)
    return terminal

## 7. EnergySource — the grid connection

The `EnergySource` represents the connection to the bulk grid (the "slack bus"
or infinite source). It connects to `bus_1` with a single terminal.

In [9]:
grid = cim.EnergySource(name='grid', EquipmentContainer=feeder)
grid.BaseVoltage = base_12kv
grid.nominalVoltage = cim.Voltage(12.47, 'kV')
grid.activePower = cim.ActivePower(0, 'MW')      # scheduled active power
grid.reactivePower = cim.ReactivePower(0, 'MVAr')
network.add_to_graph(grid)

connect(grid, buses['bus_1'], sequence_number=1)

grid.pprint()

{
    "@id": "44f4b413-98c3-466b-aace-a52642f8892f",
    "@type": "EnergySource",
    "name": "grid",
    "EquipmentContainer": {
        "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
        "@type": "Feeder"
    },
    "BaseVoltage": {
        "@id": "f314aedf-4d5b-4e11-a097-6b7a3fbadf3e",
        "@type": "BaseVoltage"
    },
    "Terminals": [
        {
            "@id": "8a3e131b-dd2d-4761-8c0b-ea37e7676bbc",
            "@type": "Terminal"
        }
    ],
    "activePower": "0.0",
    "nominalVoltage": "12470.0",
    "reactivePower": "0.0"
}


## 8. ACLineSegments — the branches

Lines connect two buses, so each `ACLineSegment` gets **two** terminals
(sequenceNumber 1 and 2). We define impedance directly with the positive-sequence
`r`/`x` attributes (in ohms) and a `length`.

We build the three line segments with a small loop.

In [10]:
line_specs = [
    # (name,       from_bus, to_bus,  length_ft, r_ohm,  x_ohm)
    # Impedances ~ 0.30 + j0.60 ohm/mile (typical 4/0 ACSR overhead) x length
    ('line_1_2',  'bus_1',  'bus_2',  1000,      0.0568, 0.1136),
    ('line_2_3',  'bus_2',  'bus_3',   800,      0.0455, 0.0909),
    ('line_3_4',  'bus_3',  'bus_4',   500,      0.0284, 0.0568),
]

lines = {}
for name, from_bus, to_bus, length_ft, r_ohm, x_ohm in line_specs:
    line = cim.ACLineSegment(name=name, EquipmentContainer=feeder)
    line.BaseVoltage = base_12kv
    line.length = cim.Length(length_ft, 'ft')
    line.r = cim.Resistance(r_ohm, 'ohm')
    line.x = cim.Reactance(x_ohm, 'ohm')

    network.add_to_graph(line)
    connect(line, buses[from_bus], sequence_number=1)
    connect(line, buses[to_bus],   sequence_number=2)
    lines[name] = line

lines['line_1_2'].pprint()

{
    "@id": "586881b6-f773-48c8-9c6c-04debf48dc05",
    "@type": "ACLineSegment",
    "name": "line_1_2",
    "EquipmentContainer": {
        "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
        "@type": "Feeder"
    },
    "BaseVoltage": {
        "@id": "f314aedf-4d5b-4e11-a097-6b7a3fbadf3e",
        "@type": "BaseVoltage"
    },
    "Terminals": [
        {
            "@id": "8564300e-fae2-4602-a70a-a0c5741ff55f",
            "@type": "Terminal"
        },
        {
            "@id": "e7934dd1-43be-44b4-b409-251fd02ec839",
            "@type": "Terminal"
        }
    ],
    "length": "304.79999999999995",
    "r": "0.0568",
    "x": "0.1136"
}


Notice the line now reports a `length` in meters (SI) even though we entered
feet — that's the `CIMUnit` system converting on input.

## 9. LoadBreakSwitch — a sectionalizing switch

A switch connects two buses, so like a line it has **two** terminals. CIM has
several switch types (`Breaker`, `Recloser`, `Fuse`, ...); we use a
`LoadBreakSwitch`, common on distribution feeders. Its state is set by two
booleans:

- `normalOpen` — the switch's *normal* position (open or closed)
- `open` — its *current* position in this study

We place it between `bus_4` and `bus_5`, closed, so power can reach the battery
inverter on `bus_5`. Opening it would isolate the DER.

In [11]:
switch = cim.LoadBreakSwitch(name='sw_4_5', EquipmentContainer=feeder)
switch.BaseVoltage = base_12kv
switch.normalOpen = False    # normally closed
switch.open = False          # closed in this study
switch.ratedCurrent = cim.CurrentFlow(600, 'A')
network.add_to_graph(switch)

connect(switch, buses['bus_4'], sequence_number=1)
connect(switch, buses['bus_5'], sequence_number=2)

switch.pprint()

{
    "@id": "b71fb3e2-8ef8-4004-ab9f-eb539102a1ff",
    "@type": "LoadBreakSwitch",
    "name": "sw_4_5",
    "EquipmentContainer": {
        "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
        "@type": "Feeder"
    },
    "BaseVoltage": {
        "@id": "f314aedf-4d5b-4e11-a097-6b7a3fbadf3e",
        "@type": "BaseVoltage"
    },
    "Terminals": [
        {
            "@id": "38d68f1d-b591-4d9c-87cb-27b885416a2e",
            "@type": "Terminal"
        },
        {
            "@id": "31051362-942b-41d3-95b8-aac7ad4e088d",
            "@type": "Terminal"
        }
    ],
    "normalOpen": "False",
    "open": "False",
    "ratedCurrent": "600.0"
}


## 10. EnergyConsumer — the load

A load is a single-terminal device. We attach it to `bus_2` and set its real and
reactive power demand. CIM's terminal sign convention: positive `p` is power
flowing **into** the equipment, so a consuming load has positive `p`.

In [12]:
load = cim.EnergyConsumer(name='load_a', EquipmentContainer=feeder)
load.BaseVoltage = base_12kv
load.p = cim.ActivePower(0.5, 'MW')
load.q = cim.ReactivePower(0.15, 'MVAr')
network.add_to_graph(load)

connect(load, buses['bus_2'], sequence_number=1)

load.pprint()

{
    "@id": "bcf084ea-708b-4564-ba06-fce3e5aaf323",
    "@type": "EnergyConsumer",
    "name": "load_a",
    "EquipmentContainer": {
        "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
        "@type": "Feeder"
    },
    "BaseVoltage": {
        "@id": "f314aedf-4d5b-4e11-a097-6b7a3fbadf3e",
        "@type": "BaseVoltage"
    },
    "Terminals": [
        {
            "@id": "67d2a694-96d1-4489-a9f6-b4a84d1023cd",
            "@type": "Terminal"
        }
    ],
    "p": "500000.0",
    "q": "150000.0"
}


## 11. LinearShuntCompensator — a shunt capacitor

A shunt capacitor injects reactive power to support voltage. It is a
single-terminal (shunt) device. CIM specifies the capacitor by its
**susceptance per section** (`bPerSection`, in siemens) rather than directly in
kvar — the relationship is:

$$ b = \frac{Q_{kvar} / 1000}{V_{kV}^2 \times N_{sections}} $$

We size a 150 kvar bank at `bus_3` (one section) to lift the mid-feeder voltage.

In [13]:
kvar_target = 150.0
kv = 12.47
b_per_section = 0.001 * kvar_target / (kv ** 2)   # siemens, for one section

capacitor = cim.LinearShuntCompensator(name='cap_3', EquipmentContainer=feeder)
capacitor.BaseVoltage = base_12kv
capacitor.nomU = cim.Voltage(12.47, 'kV')
capacitor.bPerSection = cim.Susceptance(b_per_section, 'S')
capacitor.maximumSections = 1
capacitor.sections = 1                              # one section in service
capacitor.phaseConnection = cim.PhaseShuntConnectionKind.Y
network.add_to_graph(capacitor)

connect(capacitor, buses['bus_3'], sequence_number=1)

capacitor.pprint()

{
    "@id": "68b196f9-e60b-4ffa-a270-2ca1581e3cf1",
    "@type": "LinearShuntCompensator",
    "name": "cap_3",
    "EquipmentContainer": {
        "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
        "@type": "Feeder"
    },
    "BaseVoltage": {
        "@id": "f314aedf-4d5b-4e11-a097-6b7a3fbadf3e",
        "@type": "BaseVoltage"
    },
    "Terminals": [
        {
            "@id": "109a12f1-8f18-4315-bd13-3504549c05d8",
            "@type": "Terminal"
        }
    ],
    "maximumSections": "1",
    "sections": "1",
    "nomU": "12470.0",
    "phaseConnection": "PhaseShuntConnectionKind.Y",
    "bPerSection": "0.0009646246420438721"
}


## 12. Battery inverter — PowerElectronicsConnection + BatteryUnit

A battery-backed inverter is modeled in two parts:

- **`PowerElectronicsConnection`** (PEC) — the inverter itself. It is the
  `ConductingEquipment` that connects to the AC network with a single terminal,
  and carries the AC ratings (`ratedS`, `ratedU`, `maxQ`/`minQ`).
- **`BatteryUnit`** — the DC source behind the inverter. It carries the energy
  storage attributes (`ratedE`, `storedE`, `batteryState`) and is associated to
  the PEC via `PowerElectronicsUnit`.

The DC side has no terminals — only the inverter connects to the AC grid.

In [14]:
# The inverter (AC side)
inverter = cim.PowerElectronicsConnection(name='battery_inverter', EquipmentContainer=feeder)
inverter.BaseVoltage = base_12kv
inverter.ratedS = cim.ApparentPower(0.25, 'MVA')
inverter.ratedU = cim.Voltage(12.47, 'kV')
inverter.maxQ = cim.ReactivePower(0.10, 'MVAr')
inverter.minQ = cim.ReactivePower(-0.10, 'MVAr')
network.add_to_graph(inverter)

# Connect the inverter to bus_5 (beyond the switch)
connect(inverter, buses['bus_5'], sequence_number=1)

# The battery (DC side)
battery = cim.BatteryUnit(name='battery_a')
battery.ratedE = cim.RealEnergy(0.5, 'MWh')      # capacity
battery.storedE = cim.RealEnergy(0.3, 'MWh')     # current state of charge
battery.maxP = cim.ActivePower(0.25, 'MW')
battery.minP = cim.ActivePower(-0.25, 'MW')
battery.batteryState = cim.BatteryStateKind.discharging
network.add_to_graph(battery)

# Associate the battery with its inverter (both directions)
battery.PowerElectronicsConnection = inverter
inverter.PowerElectronicsUnit.append(battery)

inverter.pprint()

{
    "@id": "61b73ae9-d609-47bb-ab5f-dfe8fd6ff2ee",
    "@type": "PowerElectronicsConnection",
    "name": "battery_inverter",
    "EquipmentContainer": {
        "@id": "3fb19ba9-06a7-4906-a18c-368be6d31a05",
        "@type": "Feeder"
    },
    "BaseVoltage": {
        "@id": "f314aedf-4d5b-4e11-a097-6b7a3fbadf3e",
        "@type": "BaseVoltage"
    },
    "Terminals": [
        {
            "@id": "904b60c6-2b83-422e-9951-92ece678e1d8",
            "@type": "Terminal"
        }
    ],
    "maxQ": "100000.0",
    "minQ": "-100000.0",
    "ratedS": "250000.0",
    "ratedU": "12470.0",
    "PowerElectronicsUnit": [
        {
            "@id": "26ac8e45-c572-4595-8f8a-249ccf6dc615",
            "@type": "BatteryUnit"
        }
    ]
}


We can read the battery's state of charge by walking from the inverter to its
unit — the same traversal an analysis tool would use.

In [15]:
for unit in inverter.PowerElectronicsUnit:
    if isinstance(unit, cim.BatteryUnit):
        soc = float(unit.storedE) / float(unit.ratedE)
        print(f'{unit.name}: state of charge = {soc:.0%}')

battery_a: state of charge = 60%


## 13. Inspect the assembled model

The graph is keyed by class. We can list how many of each object we created.

In [16]:
for cim_class in [cim.Feeder, cim.Substation, cim.BaseVoltage,
                  cim.ConnectivityNode, cim.Terminal, cim.ACLineSegment,
                  cim.LoadBreakSwitch, cim.LinearShuntCompensator,
                  cim.EnergySource, cim.EnergyConsumer,
                  cim.PowerElectronicsConnection, cim.BatteryUnit]:
    count = len(network.graph.get(cim_class, {}))
    print(f'{count:>2}  {cim_class.__name__}')

 1  Feeder
 1  Substation
 1  BaseVoltage
 5  ConnectivityNode
12  Terminal
 3  ACLineSegment
 1  LoadBreakSwitch
 1  LinearShuntCompensator
 1  EnergySource
 1  EnergyConsumer
 1  PowerElectronicsConnection
 1  BatteryUnit


### Trace the connectivity

Walk every bus and report what equipment is attached — this verifies the
terminals were wired correctly.

In [17]:
for name, node in buses.items():
    connected = [t.ConductingEquipment.__class__.__name__ for t in node.Terminals]
    print(f'{name}: {connected}')

bus_1: ['EnergySource', 'ACLineSegment']
bus_2: ['ACLineSegment', 'ACLineSegment', 'EnergyConsumer']
bus_3: ['ACLineSegment', 'ACLineSegment', 'LinearShuntCompensator']
bus_4: ['ACLineSegment', 'LoadBreakSwitch']
bus_5: ['LoadBreakSwitch', 'PowerElectronicsConnection']


## 14. Visualize the actual objects

The UML diagrams earlier showed the *class* relationships. Now we render the
**actual instances** we built and their links. `get_mermaid_path()` starts a
diagram from a root object and follows an attribute path; `add_mermaid_path()`
grafts on additional branches.

First, line `line_1_2` with both of its terminals, the buses they connect to,
and its containing feeder:

In [18]:
line = lines['line_1_2']

diagram = utils.get_mermaid_path(line, ['Terminals', '[0]', 'ConnectivityNode'])
diagram = utils.add_mermaid_path(line, ['Terminals', '[1]', 'ConnectivityNode'], diagram)
diagram = utils.add_mermaid_path(line, 'EquipmentContainer', diagram)
Mermaid(diagram)

The switch `sw_4_5` looks structurally like a line — two terminals bridging
`bus_4` and `bus_5`:

In [19]:
diagram = utils.get_mermaid_path(switch, ['Terminals', '[0]', 'ConnectivityNode'])
diagram = utils.add_mermaid_path(switch, ['Terminals', '[1]', 'ConnectivityNode'], diagram)
Mermaid(diagram)

The battery inverter — showing both the DC side (`PowerElectronicsUnit` →
`BatteryUnit`) and the AC side (`Terminals` → `ConnectivityNode` at `bus_5`):

In [20]:
diagram = utils.get_mermaid_path(inverter, ['PowerElectronicsUnit', '[0]'])
diagram = utils.add_mermaid_path(inverter, ['Terminals', '[0]', 'ConnectivityNode'], diagram)
Mermaid(diagram)

Finally, a bus-centric view: `bus_2` and every piece of equipment attached to
it (the two line segments and the load):

In [21]:
bus_2 = buses['bus_2']

diagram = utils.get_mermaid_path(bus_2, ['Terminals', '[0]', 'ConductingEquipment'])
for i in range(1, len(bus_2.Terminals)):
    diagram = utils.add_mermaid_path(
        bus_2, ['Terminals', f'[{i}]', 'ConductingEquipment'], diagram)
Mermaid(diagram)

## 15. Export to CIM XML

Finally, write the whole graph out as a CIM RDF/XML file. We use
`cimgraph.utils.write_xml()`, which serializes every object in the graph.

The CIM profile spreads attributes across several **namespaces** (the core
`cim` namespace plus a handful of extensions). `write_xml` needs the prefix for
each namespace it encounters, so we pass the full set explicitly.

In [22]:
namespaces = {
    'cim':    'http://iec.ch/TC57/CIM100#',
    'rdf':    'http://www.w3.org/1999/02/22-rdf-syntax-ns#',
    'eu':     'http://iec.ch/TC57/CIM100-European#',
    'nc':     'http://entsoe.eu/ns/nc#',
    'gb':     'http://GB/placeholder/ext#',
    'gapps':  'http://gridappsd.org/CIM/extension#',
    'cim101': 'http://iec.ch/TC57/CIM101/draft#',
}

utils.write_xml(network, 'hand_built_model.xml', namespaces=namespaces)
print('Wrote hand_built_model.xml')

Wrote hand_built_model.xml


### Read it back into a new model

Now load the file we just wrote into a **fresh** `FeederModel` and confirm the
objects survived the round trip. We compare object counts per class against the
model we built in memory.

> The reloaded model populates objects lazily — attribute values come back, but
> reverse-association lists (like `Terminals`) aren't expanded until you traverse
> the graph. For inspecting connectivity interactively, keep working with the
> `network` we built above (it is fully linked). The reload here confirms the
> **export** is complete and correct.

In [23]:
reloaded = FeederModel(connection=XMLFile('hand_built_model.xml'), container=None)

for cim_class in [cim.Feeder, cim.Substation, cim.BaseVoltage,
                  cim.ConnectivityNode, cim.Terminal, cim.ACLineSegment,
                  cim.EnergySource, cim.EnergyConsumer, cim.LoadBreakSwitch,
                  cim.LinearShuntCompensator,
                  cim.PowerElectronicsConnection, cim.BatteryUnit]:
    built = len(network.graph.get(cim_class, {}))
    loaded = len(reloaded.graph.get(cim_class, {}))
    status = 'ok' if built == loaded else 'MISMATCH'
    print(f'{cim_class.__name__:>28}: built={built:>2}  reloaded={loaded:>2}  [{status}]')

                      Feeder: built= 1  reloaded= 1  [ok]
                  Substation: built= 1  reloaded= 1  [ok]
                 BaseVoltage: built= 1  reloaded= 1  [ok]
            ConnectivityNode: built= 5  reloaded= 5  [ok]
                    Terminal: built=12  reloaded=12  [ok]
               ACLineSegment: built= 3  reloaded= 3  [ok]
                EnergySource: built= 1  reloaded= 1  [ok]
              EnergyConsumer: built= 1  reloaded= 1  [ok]
             LoadBreakSwitch: built= 1  reloaded= 1  [ok]
      LinearShuntCompensator: built= 1  reloaded= 1  [ok]
  PowerElectronicsConnection: built= 1  reloaded= 1  [ok]
                 BatteryUnit: built= 1  reloaded= 1  [ok]


And confirm the attribute values came through — list the reloaded lines with
their impedance and length:

In [24]:
for line in reloaded.list_by_class(cim.ACLineSegment):
    print(f'{line.name}: length = {float(line.length):.1f} m, '
          f'r = {float(line.r):.2f} ohm, x = {float(line.x):.2f} ohm')

line_2_3: length = 243.8 m, r = 0.05 ohm, x = 0.09 ohm
line_1_2: length = 304.8 m, r = 0.06 ohm, x = 0.11 ohm
line_3_4: length = 152.4 m, r = 0.03 ohm, x = 0.06 ohm


## 16. Export to OpenDSS and solve a power flow

A CIM model describes the *network*; to compute voltages and power flows we hand
it to an analysis engine. Here we export to **OpenDSS** using the CIMHub
`cim_to_dss` exporter, then solve with `opendssdirect`.

> This section needs two extra packages: `cimhub-opendss` (the exporter) and
> `opendssdirect-py` (the solver). They are already installed in this demo's
> environment.

In [25]:
from pathlib import Path
import tempfile

from cimhub_opendss.exporter.cim_to_dss import cim_to_dss
import opendssdirect as dss

### Export the feeder to DSS files

`cim_to_dss(network, output_dir)` writes a set of `.dss` files — one per
equipment category — plus a `Master.dss` that ties them together. It returns a
dict of `{filename: [commands]}` so we can see exactly what was generated.

In [26]:
output_dir = Path(tempfile.mkdtemp(prefix='demo_dss_'))
dss_files = cim_to_dss(network, output_dir)

for filename, commands in dss_files.items():
    print(f'### {filename}')
    for command in commands:
        print(f'  {command}')
    print()

### Switches.dss
  new Line.sw_4_5 bus1=bus_5 bus2=bus_4 phases=3 Switch=Yes

### Lines.dss
  new Line.line_2_3 bus1=bus_2 bus2=bus_3 length=243.83999999999997 r1=0.0455 x1=0.0909 units=m
  new Line.line_1_2 bus1=bus_1 bus2=bus_2 length=304.79999999999995 r1=0.0568 x1=0.1136 units=m
  new Line.line_3_4 bus1=bus_3 bus2=bus_4 length=152.39999999999998 r1=0.0284 x1=0.0568 units=m

### Capacitors.dss
  new Capacitor.cap_3 bus1=bus_3 kvar=[149.99999999999997] kv=12.47 conn=wye Numsteps=1 states=[1]

### Loads.dss
  new Load.load_a bus1=bus_2 kV=12.47 kW=500.0 pf=0.9578262852211513 model=1 kvar=150.0

### Vsources.dss
  New Circuit.grid bus1=bus_1 basekv=12.47
  Edit Vsource.grid pu=1.0 angle=0.0

### Storage.dss
  new Storage.battery_inverter bus1=bus_4 kv=12.47 kVA=250.0 kvarMax=100.0 kvarMaxAbs=100.0 kWrated=250.0 kWhrated=500.0 kWhstored=300.0 State=discharging



Each CIM object became its native DSS element:

| CIM | DSS |
|-----|-----|
| `EnergySource` | `Circuit` / `Vsource` |
| `ACLineSegment` | `Line` |
| `LoadBreakSwitch` | `Line` (`Switch=Yes`) |
| `EnergyConsumer` | `Load` |
| `LinearShuntCompensator` | `Capacitor` |
| `PowerElectronicsConnection` + `BatteryUnit` | `Storage` |

### Solve the power flow

Compile the generated `Master.dss` and solve.

In [27]:
master = output_dir / 'Master.dss'

dss.Text.Command('Clear')
dss.Text.Command(f'Compile [{master}]')
dss.Text.Command('Solve')

print('Converged:', dss.Solution.Converged())
total_p, total_q = dss.Circuit.TotalPower()  # power into the circuit (kW, kVAr)
print(f'Total source power: {-total_p:.1f} kW, {-total_q:.1f} kVAr')
print(f'Buses: {dss.Circuit.NumBuses()}, Elements: {dss.Circuit.NumCktElements()}')

Converged: True
Total source power: 265.6 kW, 29.6 kVAr
Buses: 5, Elements: 8


### Bus voltages

Read the per-unit voltage magnitude at every bus. A healthy distribution feeder
sits near 1.0 pu, sagging slightly toward the end of the line under load.

In [28]:
print(f'{"Bus":8s}  {"V (pu)":>8s}')
print('-' * 20)
for name in dss.Circuit.AllBusNames():
    dss.Circuit.SetActiveBus(name)
    v_pu = dss.Bus.puVmagAngle()[0]   # magnitude of the first phase
    print(f'{name:8s}  {v_pu:8.4f}')

Bus         V (pu)
--------------------
bus_1       1.0000
bus_5       1.0105
bus_4       1.0105
bus_2       0.9654
bus_3       1.0037


### Element power flows

The active power at the first terminal of each element. By OpenDSS convention,
power flowing **out of** an element into the circuit is negative — so the source
(`Vsource`) and the discharging battery (`Storage`) show negative P, while the
load (`Load`) draws positive P. The load is `model=1` (constant power), so it
pulls its full 500 kW regardless of the voltage sag. The closed switch
(`Line.sw_4_5`, zero impedance) and the reactive-only capacitor
(`Capacitor.cap_3`) both show ~0 kW.

In [29]:
print(f'{"Element":24s}  {"P (kW)":>10s}')
print('-' * 38)
for name in dss.Circuit.AllElementNames():
    dss.Circuit.SetActiveElement(name)
    powers = dss.CktElement.Powers()          # [P1, Q1, P2, Q2, ...] per phase
    n_phases = dss.CktElement.NumPhases()
    p_total = sum(powers[i * 2] for i in range(n_phases))
    print(f'{name:24s}  {p_total:10.2f}')

Element                       P (kW)
--------------------------------------
Vsource.source               -265.57
Line.sw_4_5                    -0.00
Line.line_2_3                -242.38
Line.line_1_2                 265.57
Line.line_3_4                -248.30
Capacitor.cap_3                 0.00
Load.load_a                   500.00
Storage.battery_inverter     -250.00


## Summary

You built a complete CIM network model by hand:

1. **Set the profile** and created a graph backed by an `XMLFile`.
2. **Containers** (`Feeder`, `Substation`) to organize and name equipment.
3. A shared **`BaseVoltage`** using `CIMUnit` (never hand-converting units).
4. **Buses** as `ConnectivityNode` objects.
5. A **`connect()` helper** encapsulating the Terminal wiring pattern.
6. An **`EnergySource`** grid connection, three **`ACLineSegment`** branches, a
   **`LoadBreakSwitch`**, an **`EnergyConsumer`** load, a
   **`LinearShuntCompensator`** capacitor, and a
   **`PowerElectronicsConnection` + `BatteryUnit`** inverter.
7. **Visualized** both the class hierarchy (`get_mermaid`) and the actual object
   instances (`get_mermaid_path` / `add_mermaid_path`).
8. **Exported** to CIM XML with `utils.write_xml()` and **reloaded** into a new
   `FeederModel` to verify.
9. **Exported to OpenDSS** with `cim_to_dss`, **solved** the power flow, and read
   back bus voltages and element power flows.

### Key takeaways
- Every object must be added with `network.add_to_graph(...)`.
- Equipment connects to buses through `Terminal` objects; branches use two
  terminals (sequenceNumber 1 and 2), shunt devices use one.
- Use `CIMUnit` constructors (`cim.Voltage`, `cim.ActivePower`, ...) on input and
  `.to(unit)` on output — never multiply by `1e3`/`1e6` by hand.
- Associations are directional; set the reverse side when you need to traverse
  both ways.